# NASA CMR: Consortiums, Data Centers and EOSDIS Metadata Associations

**Questions:**
1. How many unique consortiums are there (`collections.items.consortiums`)?
2. Which data centers (`collections.items.dataCenter`) fall under each consortium?
3. How many collections does each data center have, and how many does each consortium have?
4. How many collections does each data center have under the **EOSDIS** consortium?
5. List every collection from question 4 with its data center.
6. Split the question 5 list by whether the collection has `metadataAssociations` (saved as 2 separate files).
7. Save the results as JSON and CSV in `cmr_data_fetch/`.

**Environment:** the `uv_metadata` uv virtual environment (`.venv`, Python 3.14, with `requests` and `pandas`).

## Step 1 - Check the Earthdata MCP and the GraphQL query

* `earthdata_get_collections(limit=1)` gives **50,000+** collections in total. The MCP tools don't expose `consortiums`, `dataCenter` or `metadataAssociations`, so this notebook queries **CMR GraphQL** (`https://graphql.earthdata.nasa.gov/api`) directly. It's the official GraphQL interface on the same CMR index the MCP uses.
* The query looks like the following (`cursor` is for paging):

```graphql
query Collections($collectionParams: CollectionsInput) {
  collections(params: $collectionParams) {
    count
    cursor
    items {
      abstract
      conceptId
      consortiums
      dataCenter
      doi
      entryId
      entryTitle
      metadataAssociations
      processingLevelId
      platforms
      relatedUrls
      scienceKeywords
      shortName
      provider
      version
    }
  }
}
```

**What the test queries showed:**
* `consortiums` is a **list** and can be `null` (e.g. all SCIOPS records). A collection can belong to several consortiums (e.g. `["CEOS","FEDEO"]`), so **per-consortium counts add up to more than the total number of collections**. Collections with no consortium are reported as `(none)`.
* `metadataAssociations` is the UMM-C `MetadataAssociations` field: links to other metadata records, e.g. `{"type": "SCIENCE ASSOCIATED", "entryId": "..."}`. GraphQL returns either `null` or a non-empty list. A collection **has** metadata associations if the list is non-empty.
* `doi`, `platforms`, `relatedUrls` and `scienceKeywords` are nested objects/lists straight from UMM-C (e.g. `doi` is `{"doi": "..."}` or `{"missingReason": "...", "explanation": "..."}` when a collection has no DOI). `processingLevelId` is flattened by GraphQL from UMM-C's `ProcessingLevel.Id` and can be `null`.
* GraphQL can filter by consortium on the server (`consortium: "EOSDIS"`). The notebook uses this as an independent check on the counts calculated here. GraphQL can't filter on `metadataAssociations`, so the step 6 split is checked against the CMR Search API `umm_json` records instead.

**Robustness:** in an earlier run, paging through all collections in one sequence hit a persistent `CMR_ERROR: Unknown Error`. So this notebook pages **one provider at a time**. If GraphQL keeps failing for a provider, it re-fetches that provider from the CMR Search API. `collections.json` only provides `consortiums` and `data_center` in the same shape GraphQL uses - everything else (`abstract`, `doi`, `metadataAssociations`, `processingLevelId`, `platforms`, `relatedUrls`, `scienceKeywords`, `version`) is read from `collections.umm_json` instead and converted from UMM's PascalCase keys to GraphQL's camelCase. A failed provider therefore isn't wrongly counted as having no associations or missing these fields.

**Progress log:** the notebook is executed in the background with `nbconvert`. Progress is written to `run.log` so a long run can be followed.

In [1]:
import json
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

GRAPHQL_URL = "https://graphql.earthdata.nasa.gov/api"
CMR_SEARCH = "https://cmr.earthdata.nasa.gov/search"
PAGE_SIZE = 2000
OUT_NAME = "cmr_data_fetch"

OUT_DIR = Path.cwd() if Path.cwd().name == OUT_NAME else Path.cwd() / OUT_NAME
OUT_DIR.mkdir(exist_ok=True)
LOG = OUT_DIR / "run.log"
LOG.write_text("", encoding="utf-8")


def log(msg):
    """Print and append to run.log (nbconvert only saves cell output at the end)."""
    print(msg, flush=True)
    with LOG.open("a", encoding="utf-8") as f:
        f.write(f"{datetime.now():%H:%M:%S} {msg}\n")


QUERY = """
query Collections($collectionParams: CollectionsInput) {
  collections(params: $collectionParams) {
    count
    cursor
    items {
      abstract
      conceptId
      consortiums
      dataCenter
      doi
      entryId
      entryTitle
      metadataAssociations
      processingLevelId
      platforms
      relatedUrls
      scienceKeywords
      shortName
      provider
      version
    }
  }
}
"""
session = requests.Session()
log(f"Output directory: {OUT_DIR}")

Output directory: /Users/eleu/dev/esdis/metadata-quality-skills-clone/cmr_data_fetch


## Step 2 - Helpers: GraphQL paging, the fallback, and the provider list

In [2]:
def post_graphql(params, retries=4):
    """Return the `collections` block, or None if GraphQL keeps failing."""
    err = None
    for attempt in range(retries):
        try:
            r = session.post(
                GRAPHQL_URL,
                timeout=300,
                json={"query": QUERY, "variables": {"collectionParams": params}}
            )
            body = r.json()
            if r.status_code == 200 and (body.get("data") or {}).get("collections"):
                return body["data"]["collections"]
            else:
                err = (body.get("errors") or [{}])[0].get("message")
        except (requests.RequestException, ValueError) as e:
            err = repr(e)
        time.sleep(2 ** attempt)
    log(f"   GraphQL failed for {params}: {err}")
    return None


def graphql_count(**filters):
    data = post_graphql({**filters, "limit": 0})
    return None if data is None else data["count"]


def fetch_provider_graphql(provider):
    params = {"provider": provider, "limit": PAGE_SIZE}
    items = []
    total = None
    while True:
        data = post_graphql(params)
        if data is None:
            return None
        total = data["count"] if total is None else total
        batch = data["items"] or []
        items.extend(batch)
        if not batch or len(items) >= total or not data.get("cursor"):
            return items
        params = {"provider": provider, "limit": PAGE_SIZE, "cursor": data["cursor"]}


def search_api_pages(fmt, params):
    """Yield pages of CMR Search API results (search-after paging)."""
    headers, seen = {}, 0
    while True:
        for attempt in range(5):
            r = session.get(
                f"{CMR_SEARCH}/collections.{fmt}",
                headers=headers,
                timeout=300,
                params={**params, "page_size": PAGE_SIZE}
            )
            if r.status_code == 200:
                break
            time.sleep(2 ** attempt)
        r.raise_for_status()
        body = r.json()
        batch = body["feed"]["entry"] if fmt == "json" else body["items"]
        seen += len(batch)
        yield batch
        sa = r.headers.get("CMR-Search-After")
        if not batch or not sa or seen >= int(r.headers["CMR-Hits"]):
            return
        headers = {"CMR-Search-After": sa}


def camel_case_keys(obj):
    """Recursively convert UMM PascalCase keys to GraphQL's camelCase (e.g. 'URL' -> 'url')."""
    if isinstance(obj, dict):
        return {k[0].lower() + k[1:]: camel_case_keys(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [camel_case_keys(v) for v in obj]
    return obj


def normalize_umm_associations(assocs):
    """UMM-C MetadataAssociations -> the GraphQL shape (camelCase keys, None when empty)."""
    if not assocs:
        return None
    return camel_case_keys(assocs)


def umm_associations(params):
    """conceptId -> MetadataAssociations (GraphQL shape) from collections.umm_json."""
    out = {}
    for batch in search_api_pages("umm_json", params):
        for e in batch:
            out[e["meta"]["concept-id"]] = normalize_umm_associations(e["umm"].get("MetadataAssociations"))
    return out


def umm_extra_fields(params):
    """conceptId -> the fields collections.json doesn't expose (GraphQL shape), from collections.umm_json."""
    out = {}
    for batch in search_api_pages("umm_json", params):
        for e in batch:
            u = e["umm"]
            out[e["meta"]["concept-id"]] = {
                "abstract": u.get("Abstract"),
                "doi": camel_case_keys(u.get("DOI")),
                "metadataAssociations": normalize_umm_associations(u.get("MetadataAssociations")),
                "processingLevelId": (u.get("ProcessingLevel") or {}).get("Id"),
                "platforms": camel_case_keys(u.get("Platforms")) or None,
                "relatedUrls": camel_case_keys(u.get("RelatedUrls")) or None,
                "scienceKeywords": camel_case_keys(u.get("ScienceKeywords")) or None,
                "version": u.get("Version"),
            }
    return out


def fetch_provider_search_api(provider):
    """Fallback: collections.json (consortiums, data_center) + umm_json (everything collections.json doesn't have)."""
    extra = umm_extra_fields({"provider": provider})
    items = []
    for batch in search_api_pages("json", {"provider": provider}):
        for e in batch:
            u = extra.get(e.get("id"), {})
            items.append(
                {
                    "abstract": u.get("abstract"),
                    "conceptId": e.get("id"),
                    "consortiums": e.get("consortiums"),
                    "dataCenter": e.get("data_center"),
                    "doi": u.get("doi"),
                    "entryId": f"{e.get('short_name')}_{e.get('version_id')}",
                    "entryTitle": e.get("title"),
                    "metadataAssociations": u.get("metadataAssociations"),
                    "processingLevelId": u.get("processingLevelId"),
                    "platforms": u.get("platforms"),
                    "relatedUrls": u.get("relatedUrls"),
                    "scienceKeywords": u.get("scienceKeywords"),
                    "shortName": e.get("short_name"),
                    "provider": e.get("data_center"),
                    "version": u.get("version"),
                }
            )
    return items


# All CMR providers (public list). Providers with no collections simply return 0 items.
r = session.get("https://cmr.earthdata.nasa.gov/ingest/providers", timeout=120)
r.raise_for_status()
providers = sorted(p["provider-id"] for p in r.json())
total = graphql_count()
log(f"Total collections (GraphQL count): {total:,}")
log(f"Providers to query: {len(providers)}")

Total collections (GraphQL count): 53,652
Providers to query: 60


## Step 3 - Fetch every collection, one provider at a time

The previous run took about 8 minutes for this step. Progress is written to `run.log`.

In [3]:
t0 = time.time()
items, source_by_provider = [], {}
for i, prov in enumerate(providers, 1):
    got = fetch_provider_graphql(prov)
    src = "graphql"
    if got is None:
        got = fetch_provider_search_api(prov)
        src = "search_api_fallback"
    source_by_provider[prov] = src
    items.extend(got)
    if got:
        log(f"[{i:>3}/{len(providers)}] {prov:<16} {len(got):>7,}  ({src})  {time.time() - t0:5.0f} s")
retrieved_at = datetime.now(timezone.utc).isoformat()
log(f"Retrieved {len(items):,} items in {time.time() - t0:.0f} s")

[  1/60] AMD_KOPRI          1,855  (graphql)     25 s
[  2/60] AMD_USAPDC           966  (graphql)     36 s
[  3/60] ASF                  177  (graphql)     39 s
[  4/60] ASIPS                 15  (graphql)     45 s
   GraphQL failed for {'provider': 'AU_AADC', 'limit': 2000}: None
[  5/60] AU_AADC            2,995  (search_api_fallback)    186 s
[  6/60] CCMEO                  3  (graphql)    188 s
[  7/60] CDDIS                204  (graphql)    191 s
   GraphQL failed for {'provider': 'CEOS_EXTRA', 'limit': 2000}: None
[  8/60] CEOS_EXTRA         2,154  (search_api_fallback)    329 s
[  9/60] CSDA                  23  (graphql)    333 s
[ 13/60] ENVIDAT              731  (graphql)    352 s
[ 14/60] ESA                  188  (graphql)    368 s
[ 15/60] ESDIS                327  (graphql)    373 s
[ 16/60] EUMETSAT              36  (graphql)    380 s
[ 17/60] FEDEO                346  (graphql)    392 s
   GraphQL failed for {'provider': 'GES_DISC', 'limit': 2000}: None
[ 19/60] GES_DI

In [4]:
df = pd.DataFrame(items).drop_duplicates("conceptId").reset_index(drop=True)
for col in ["shortName", "entryId", "entryTitle"]:
    df[col] = df[col].str.strip()   # some SCIOPS shortNames contain stray newlines/spaces
df["consortiums"] = df["consortiums"].apply(lambda v: sorted(set(v)) if isinstance(v, list) and v else [])

# Normalise: a collection has metadata associations only if the list is non-empty
df["metadataAssociations"] = df["metadataAssociations"].apply(lambda v: v if isinstance(v, list) and v else None)

# Same normalisation for the other list-valued fields added to the query
for col in ["platforms", "relatedUrls", "scienceKeywords"]:
    df[col] = df[col].apply(lambda v: v if isinstance(v, list) and v else None)

fallback = [p for p, s in source_by_provider.items() if s != "graphql"]
print(f"Total collections (GraphQL count): {total:,}")
print(f"Unique collections retrieved:      {len(df):,}  ({len(df) / total:.2%})")
print(f"Fallback used for: {fallback or 'none'}")
print(f"dataCenter == provider for all rows: {(df['dataCenter'] == df['provider']).all()}")
print(f"Collections with no consortium:      {(df['consortiums'].str.len() == 0).sum():,}")
print(f"Collections with metadataAssociations: {df['metadataAssociations'].notna().sum():,}")
print(df.head())

Total collections (GraphQL count): 53,652
Unique collections retrieved:      53,652  (100.00%)
Fallback used for: ['AU_AADC', 'CEOS_EXTRA', 'GES_DISC', 'GHRSSTCWIC', 'LARC_CLOUD', 'NOAA_NCEI', 'ORNL_CLOUD', 'POCLOUD', 'SCIOPS']
dataCenter == provider for all rows: True
Collections with no consortium:      12,829
Collections with metadataAssociations: 5,151
                                            abstract              conceptId  \
0  Korean Antarctic survey was conducted in north...  C2244294500-AMD_KOPRI   
1  Korean Antarctic survey carried out the fifth ...  C2244294814-AMD_KOPRI   
2  Korean Antarctic survey carried out as part of...  C2244294883-AMD_KOPRI   
3  Korean Antarctic survey carried out as part of...  C2244294924-AMD_KOPRI   
4  Korean Antarctic survey carried out as part of...  C2244294933-AMD_KOPRI   

  consortiums dataCenter                                                doi  \
0          []  AMD_KOPRI  {'missingReason': 'Unknown', 'explanation': 'I...   
1       

## Step 4 - Q1: Unique consortiums

One row per (collection, consortium) pair. Collections with no consortium are labelled `(none)`, which is **not counted** as a consortium.

In [5]:
NONE = "(none)"
pairs = df[["conceptId", "dataCenter", "consortiums"]].copy()
pairs["consortium"] = pairs["consortiums"].apply(lambda v: v or [NONE])
pairs = pairs.explode("consortium").drop(columns="consortiums")

unique_consortiums = sorted(c for c in pairs["consortium"].unique() if c != NONE)
print(f"Q1 - Number of unique consortiums: {len(unique_consortiums)}")
print(unique_consortiums)

Q1 - Number of unique consortiums: 5
['CEOS', 'CWIC', 'EOSDIS', 'FEDEO', 'GEOSS']


## Step 5 - Q2 & Q3: Data centers per consortium, and collection counts

* **Collections per consortium:** a collection is counted once in each consortium it belongs to.
* **Collections per data center:** each collection is counted exactly once, so these add up to the total.
* **Collections per (consortium, data center):** the detail table behind Q2.

In [6]:
consortium_summary = (
    pairs.groupby("consortium").agg(collection_count=("conceptId", "nunique"),
    data_center_count=("dataCenter", "nunique"),
    data_centers=("dataCenter", lambda s: sorted(s.unique()))).sort_values("collection_count", ascending=False).reset_index()
)

datacenter_summary = (
    df.assign(consortiums=df["consortiums"].apply(lambda v: v or [NONE])).groupby("dataCenter").agg(collection_count=("conceptId", "nunique"),
    consortiums=("consortiums", lambda s: sorted({c for v in s for c in v}))).sort_values("collection_count", ascending=False).reset_index()
)

consortium_datacenter = (
    pairs.groupby(["consortium", "dataCenter"])["conceptId"].nunique().rename("collection_count").reset_index().sort_values(["consortium", "collection_count"], ascending=[True, False])
)

assert datacenter_summary["collection_count"].sum() == len(df)
print(f"Unique data centers: {len(datacenter_summary)}")
print(consortium_summary[["consortium", "collection_count", "data_center_count"]])
print(datacenter_summary)

Unique data centers: 40
  consortium  collection_count  data_center_count
0       CEOS             25612                 13
1      GEOSS             14698                 26
2     (none)             12829                  5
3     EOSDIS             11376                 21
4      FEDEO               534                  2
5       CWIC               315                  6
    dataCenter  collection_count          consortiums
0    NOAA_NCEI             21476               [CEOS]
1       SCIOPS             10094      [(none), GEOSS]
2      AU_AADC              2995      [(none), GEOSS]
3   CEOS_EXTRA              2154               [CEOS]
4     GES_DISC              2049      [EOSDIS, GEOSS]
5   ORNL_CLOUD              1997      [EOSDIS, GEOSS]
6    AMD_KOPRI              1855             [(none)]
7   LARC_CLOUD              1748      [EOSDIS, GEOSS]
8         JAXA              1024               [CEOS]
9      POCLOUD              1004      [EOSDIS, GEOSS]
10  AMD_USAPDC               966

In [7]:
# Q2 - every data center under each consortium
pd.set_option("display.max_colwidth", None)
for _, row in consortium_summary.iterrows():
    sub = consortium_datacenter[consortium_datacenter["consortium"] == row["consortium"]]
    listing = ", ".join(f"{d} ({n:,})" for d, n in zip(sub["dataCenter"], sub["collection_count"]))
    print(f"{row['consortium']}  [{row['collection_count']:,} collections, {row['data_center_count']} data centers]\n   {listing}\n")

CEOS  [25,612 collections, 13 data centers]
   NOAA_NCEI (21,476), CEOS_EXTRA (2,154), JAXA (1,024), FEDEO (346), ESA (188), USGS_LTA (128), GHRSSTCWIC (112), INPE (73), ISRO (39), EUMETSAT (36), NRSCC (29), USGS_EROS (4), CCMEO (3)

GEOSS  [14,698 collections, 26 data centers]
   AU_AADC (2,985), GES_DISC (2,049), ORNL_CLOUD (1,997), LARC_CLOUD (1,748), POCLOUD (1,004), LPCLOUD (793), ENVIDAT (731), GHRC_DAAC (700), NSIDC_CPRD (636), OB_CLOUD (478), OB_DAAC (455), CDDIS (204), LAADS (196), ASF (175), LANCEMODIS (173), NSIDCV0 (127), SCIOPS (119), MLHUB (58), OMINRT (22), ASIPS (15), ISRO (13), LANCEAMSR2 (7), NCCS (7), USGS_EROS (3), LPDAAC_ECS (2), ESDIS (1)

(none)  [12,829 collections, 5 data centers]
   SCIOPS (9,975), AMD_KOPRI (1,855), AMD_USAPDC (966), CSDA (23), AU_AADC (10)

EOSDIS  [11,376 collections, 21 data centers]
   GES_DISC (2,049), ORNL_CLOUD (1,997), LARC_CLOUD (1,748), POCLOUD (1,004), LPCLOUD (793), GHRC_DAAC (700), NSIDC_CPRD (636), OB_CLOUD (478), OB_DAAC (455),

## Step 6 - Q4: Data centers under the EOSDIS consortium

In [8]:
eosdis = (
    consortium_datacenter[consortium_datacenter["consortium"] == "EOSDIS"].drop(columns="consortium").reset_index(drop=True)
)

# What share of each data center's collections is EOSDIS?
eosdis = eosdis.merge(
    datacenter_summary[["dataCenter", "collection_count"]].rename(columns={"collection_count": "total_collections_in_data_center"}),
    on="dataCenter"
)
eosdis["pct_of_data_center"] = (100 * eosdis["collection_count"] / eosdis["total_collections_in_data_center"]).round(1)
print(f"EOSDIS: {eosdis['collection_count'].sum():,} collections across {len(eosdis)} data centers")
print(eosdis)

EOSDIS: 11,376 collections across 21 data centers
    dataCenter  collection_count  total_collections_in_data_center  \
0     GES_DISC              2049                              2049   
1   ORNL_CLOUD              1997                              1997   
2   LARC_CLOUD              1748                              1748   
3      POCLOUD              1004                              1004   
4      LPCLOUD               793                               793   
5    GHRC_DAAC               700                               700   
6   NSIDC_CPRD               636                               636   
7     OB_CLOUD               478                               478   
8      OB_DAAC               455                               455   
9      NSIDCV0               328                               328   
10       ESDIS               327                               327   
11       CDDIS               204                               204   
12       LAADS               196        

## Step 7 - Check the counts against CMR's own consortium filter

For each consortium, ask GraphQL to count it on the server (`consortium: X`) and compare that with the count calculated here. A small difference is possible if records change in CMR during the run.

In [9]:
check = consortium_summary[consortium_summary["consortium"] != NONE][["consortium", "collection_count"]].copy()
check["graphql_server_count"] = [graphql_count(consortium=c) for c in check["consortium"]]
check["match"] = check["collection_count"] == check["graphql_server_count"]
eosdis_server = {dc: graphql_count(consortium="EOSDIS", provider=dc) for dc in eosdis["dataCenter"]}
eosdis["graphql_server_count"] = eosdis["dataCenter"].map(eosdis_server)
log(f"Consortium totals matching server count: {check['match'].sum()}/{len(check)}")
log(f"EOSDIS data-center counts matching server: {(eosdis['collection_count'] == eosdis['graphql_server_count']).sum()}/{len(eosdis)}")
print(check)

Consortium totals matching server count: 5/5
EOSDIS data-center counts matching server: 21/21
  consortium  collection_count  graphql_server_count  match
0       CEOS             25612                 25612   True
1      GEOSS             14698                 14698   True
3     EOSDIS             11376                 11376   True
4      FEDEO               534                   534   True
5       CWIC               315                   315   True


## Step 8 - Q5: Every EOSDIS collection and its data center

These are all collections whose `consortiums` list contains `EOSDIS`. Each is identified by `shortName`, `entryId` (shortName_version), `entryTitle` and `conceptId`, and sorted by data center and then short name. The per-data-center counts must match question 4.

In [10]:
def as_assoc_list(v):
    return v if isinstance(v, list) and v else None


EXTRA_FIELDS = ["abstract", "doi", "processingLevelId", "platforms", "relatedUrls", "scienceKeywords", "version"]

eosdis_coll = (
    df[df["consortiums"].apply(lambda v: "EOSDIS" in v)][
        ["dataCenter", "shortName", "entryId", "entryTitle", "conceptId", "consortiums", "metadataAssociations", *EXTRA_FIELDS]
    ].sort_values(["dataCenter", "shortName", "entryId"]).reset_index(drop=True)
)
eosdis_coll["metadataAssociations"] = eosdis_coll["metadataAssociations"].apply(as_assoc_list)
eosdis_coll["has_metadata_associations"] = eosdis_coll["metadataAssociations"].apply(lambda v: v is not None)
eosdis_coll["association_count"] = eosdis_coll["metadataAssociations"].apply(lambda v: len(v) if v else 0)
eosdis_coll["association_types"] = eosdis_coll["metadataAssociations"].apply(
    lambda v: sorted({a.get("type") or "(no type)" for a in v}) if v else []
)

per_dc = eosdis_coll.groupby("dataCenter").size()
assert len(eosdis_coll) == eosdis["collection_count"].sum()
assert (per_dc.reindex(eosdis["dataCenter"]).values == eosdis["collection_count"].values).all()
print(f"Q5 - EOSDIS collections: {len(eosdis_coll):,} across {eosdis_coll['dataCenter'].nunique()} data centers")
print(f"Unique shortNames: {eosdis_coll['shortName'].nunique():,}   unique entryIds: {eosdis_coll['entryId'].nunique():,}")
print(eosdis_coll[["dataCenter", "shortName", "entryId", "entryTitle", "conceptId"]].head(10))

Q5 - EOSDIS collections: 11,376 across 21 data centers
Unique shortNames: 10,785   unique entryIds: 11,366
  dataCenter           shortName               entryId  \
0        ASF       ALOS2_L1_PSR2       ALOS2_L1_PSR2_1   
1        ASF  ALOS_AVNIR_OBS_ORI  ALOS_AVNIR_OBS_ORI_2   
2        ASF        ALOS_PSR_KMZ        ALOS_PSR_KMZ_1   
3        ASF       ALOS_PSR_L1.0       ALOS_PSR_L1.0_1   
4        ASF       ALOS_PSR_L1.1       ALOS_PSR_L1.1_1   
5        ASF       ALOS_PSR_L1.5       ALOS_PSR_L1.5_1   
6        ASF       ALOS_PSR_L2.2       ALOS_PSR_L2.2_1   
7        ASF   ALOS_PSR_RTC_HIGH   ALOS_PSR_RTC_HIGH_1   
8        ASF    ALOS_PSR_RTC_LOW    ALOS_PSR_RTC_LOW_1   
9        ASF        ARIA_S1_GUNW        ARIA_S1_GUNW_1   

                                                          entryTitle  \
0                            ALOS-2 PALSAR-2 Level 1 ScanSAR Product   
1                         ALOS AVNIR-2 Ortho Rectified Image Product   
2          KMZ for ALOS PALSAR Radiome

## Step 9 - Q6: Split EOSDIS collections by `metadataAssociations`

* **with**: `metadataAssociations` is a non-empty list.
* **without**: `metadataAssociations` is `null` or empty.

The two sets don't overlap and together make up the full question 5 list.

In [11]:
with_assoc = eosdis_coll[eosdis_coll["has_metadata_associations"]].reset_index(drop=True)
without_assoc = eosdis_coll[~eosdis_coll["has_metadata_associations"]].reset_index(drop=True)
assert len(with_assoc) + len(without_assoc) == len(eosdis_coll)
assert not set(with_assoc["conceptId"]) & set(without_assoc["conceptId"])

assoc_by_dc = (
    eosdis_coll.groupby("dataCenter").agg(
        eosdis_collections=("conceptId", "size"),
        with_metadata_associations=("has_metadata_associations", "sum")
    ).reset_index()
)
assoc_by_dc["without_metadata_associations"] = assoc_by_dc["eosdis_collections"] - assoc_by_dc["with_metadata_associations"]
assoc_by_dc["pct_with"] = (100 * assoc_by_dc["with_metadata_associations"] / assoc_by_dc["eosdis_collections"]).round(1)  # pct_with is the percent with metadataAssociations
assoc_by_dc = assoc_by_dc.sort_values(["with_metadata_associations", "eosdis_collections"], ascending=False).reset_index(drop=True)

# Association types: collections that have each type, and total association entries
type_collections = with_assoc["association_types"].explode().value_counts()
type_entries = Counter(a.get("type") or "(no type)" for v in with_assoc["metadataAssociations"] for a in v)
assoc_types = (
    pd.DataFrame({
        "collections_with_type": type_collections,
        "association_entries": pd.Series(type_entries)
    }).rename_axis("type").reset_index().sort_values("association_entries", ascending=False)
)

log(
    f"Q6 - with metadataAssociations: {len(with_assoc):,} ({len(with_assoc) / len(eosdis_coll):.1%}); "
    f"without: {len(without_assoc):,}"
)
print(assoc_by_dc)
print(assoc_types)

Q6 - with metadataAssociations: 750 (6.6%); without: 10,626
    dataCenter  eosdis_collections  with_metadata_associations  \
0   LARC_CLOUD                1748                         526   
1     GES_DISC                2049                         157   
2          ASF                 177                          41   
3   LANCEMODIS                 173                          20   
4      LPCLOUD                 793                           3   
5   LPDAAC_ECS                   2                           2   
6    GHRC_DAAC                 700                           1   
7   ORNL_CLOUD                1997                           0   
8      POCLOUD                1004                           0   
9   NSIDC_CPRD                 636                           0   
10    OB_CLOUD                 478                           0   
11     OB_DAAC                 455                           0   
12     NSIDCV0                 328                           0   
13       ESDIS  

## Step 10 - Check the split against the CMR Search API

GraphQL can't filter on `metadataAssociations`, so this step re-reads all EOSDIS collections from `collections.umm_json?consortium=EOSDIS`, a separate CMR endpoint. It then compares, for each collection:
* whether it has `MetadataAssociations`, and
* the association entries themselves, after converting UMM keys such as `EntryId` to GraphQL's `entryId`.

A small difference is possible if records change in CMR during the run.

In [12]:
t0 = time.time()
log("Step 10 - reading collections.umm_json?consortium=EOSDIS ...")
umm_assoc = umm_associations({"consortium": "EOSDIS"})
log(f"   {len(umm_assoc):,} records in {time.time() - t0:.0f} s")

gql_ids, umm_ids = set(eosdis_coll["conceptId"]), set(umm_assoc)
gql_with, umm_with = set(with_assoc["conceptId"]), {c for c, v in umm_assoc.items() if v}
gql_map = dict(zip(eosdis_coll["conceptId"], eosdis_coll["metadataAssociations"]))
common = gql_ids & umm_ids
same_content = sum(gql_map[c] == umm_assoc[c] for c in common)

split_check = {
    "search_api_eosdis_count": len(umm_ids),
    "graphql_eosdis_count": len(gql_ids),
    "search_api_with_associations": len(umm_with & common),
    "graphql_with_associations": len(gql_with & common),
    "only_in_graphql": sorted(gql_ids - umm_ids),
    "only_in_search_api": sorted(umm_ids - gql_ids),
    "with_in_graphql_only": sorted((gql_with - umm_with) & common),
    "with_in_search_api_only": sorted((umm_with - gql_with) & common),
    "identical_association_content": same_content,
    "compared": len(common),
}
split_check["split_matches"] = not split_check["with_in_graphql_only"] and not split_check["with_in_search_api_only"]
log(
    f"Split matches Search API: {split_check['split_matches']}  "
    f"(identical association content: {same_content:,}/{len(common):,})"
)
print({k: (v if not isinstance(v, list) else f"{len(v)} {v[:5]}") for k, v in split_check.items()})

Step 10 - reading collections.umm_json?consortium=EOSDIS ...
   11,376 records in 38 s
Split matches Search API: True  (identical association content: 11,376/11,376)
{'search_api_eosdis_count': 11376, 'graphql_eosdis_count': 11376, 'search_api_with_associations': 750, 'graphql_with_associations': 750, 'only_in_graphql': '0 []', 'only_in_search_api': '0 []', 'with_in_graphql_only': '0 []', 'with_in_search_api_only': '0 []', 'identical_association_content': 11376, 'compared': 11376, 'split_matches': True}


## Step 11 - Save the results (JSON + CSV)

| File | Contents |
|---|---|
| `consortiums_datacenters_associations.json` | Answers to Q1-Q6 (summaries), the query, and the checks |
| `consortium_summary.csv`, `datacenter_summary.csv`, `consortium_datacenter_counts.csv`, `eosdis_datacenter_counts.csv` | Q1-Q4 tables |
| `collections_consortiums.csv` | All collections with every queried field, including `abstract`, `doi`, `processingLevelId`, `platforms`, `relatedUrls`, `scienceKeywords` and `version` |
| `eosdis_collections.{csv,json}` | Q5: every EOSDIS collection, its data center, and the same extra fields |
| *`eosdis_collections_w_metadata_associations.{csv,json}` | Q6: EOSDIS collections **with** `metadataAssociations` |
| `eosdis_collections_wo_metadata_associations.{csv,json}` | Q6: EOSDIS collections **without** `metadataAssociations` |
| `eosdis_datacenter_associations_summary.csv` | Q6: with/without counts per data center |

In CSV files, list columns are joined with `; ` and nested fields (`metadataAssociations`, `doi`, `platforms`, `relatedUrls`, `scienceKeywords`) are stored as JSON strings. JSON files keep them as real lists/objects. The Q5/Q6 per-collection files carry the identifying columns (`BASE_COLS`), the question-specific fields, and the full set of extra queried fields (`EXTRA_FIELDS`) added in step 1.

*Only this file will be created. Uncomment the code if you need other files.

In [13]:
BASE_COLS = ["dataCenter", "shortName", "entryId", "entryTitle", "conceptId"]


def records(frame, extra=()):
    out = []
    for r in frame.to_dict(orient="records"):
        rec = {c: r[c] for c in BASE_COLS}
        for c in (*extra, *EXTRA_FIELDS):
            v = r[c]
            rec[c] = bool(v) if c == "has_metadata_associations" else int(v) if c == "association_count" else v
        out.append(rec)
    return out


def to_csv(frame, path, cols):
    f = frame[cols].copy()
    for c in ("association_types", "consortiums"):
        if c in f:
            f[c] = f[c].str.join("; ")
    for c in ("metadataAssociations", "doi", "platforms", "relatedUrls", "scienceKeywords"):
        if c in f:
            f[c] = f[c].apply(lambda v: json.dumps(v) if v else "")
    f.to_csv(path, index=False)


def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=int), encoding="utf-8")


p = lambda name: OUT_DIR / name
meta = {"source": GRAPHQL_URL, "retrieved_at_utc": retrieved_at, "consortium": "EOSDIS"}

# # ---- Q1-Q4 (same files as the consortiums run) ----
# cs = consortium_summary.copy(); cs["data_centers"] = cs["data_centers"].str.join("; ")
# cs.to_csv(p("consortium_summary.csv"), index=False)
# ds = datacenter_summary.copy(); ds["consortiums"] = ds["consortiums"].str.join("; ")
# ds.to_csv(p("datacenter_summary.csv"), index=False)
# consortium_datacenter.to_csv(p("consortium_datacenter_counts.csv"), index=False)
# eosdis.to_csv(p("eosdis_datacenter_counts.csv"), index=False)
# to_csv(df, p("collections_consortiums.csv"), list(df.columns))

# # ---- Q5 ----
# q5_cols = BASE_COLS + ["has_metadata_associations", "association_count"] + EXTRA_FIELDS
# to_csv(eosdis_coll, p("eosdis_collections.csv"), q5_cols)
# write_json(
#     p("eosdis_collections.json"),
#     {**meta, "count": len(eosdis_coll),
#     "collections": records(eosdis_coll, ["has_metadata_associations", "association_count"])}
# )

# ---- Q6 (two separate files, each as CSV + JSON) ----
with_cols = BASE_COLS + ["association_count", "association_types", "metadataAssociations"] + EXTRA_FIELDS
to_csv(
    with_assoc,
    p("eosdis_collections_w_metadata_associations.csv"),
    with_cols
)
write_json(
    p("eosdis_collections_w_metadata_associations.json"),
    {**meta, "has_metadata_associations": True, "count": len(with_assoc),
    "collections": records(with_assoc, ["association_count", "association_types", "metadataAssociations"])}
)
without_cols = BASE_COLS + EXTRA_FIELDS
to_csv(
    without_assoc,
    p("eosdis_collections_wo_metadata_associations.csv"),
    without_cols
)
write_json(
    p("eosdis_collections_wo_metadata_associations.json"),
    {**meta, "has_metadata_associations": False, "count": len(without_assoc),
    "collections": records(without_assoc)}
)
assoc_by_dc.to_csv(p("eosdis_datacenter_associations_summary.csv"), index=False)

# # ---- Summary JSON (Q1-Q6) ----
# payload = {
#     **{k: v for k, v in meta.items() if k != "consortium"},
#     "graphql_query": QUERY.strip(),
#     "total_collections_graphql_count": total,
#     "collections_retrieved": len(df),
#     "collections_without_consortium": int((df["consortiums"].str.len() == 0).sum()),
#     "fallback_providers": fallback,
#     "note": "Collections can belong to several consortiums, so per-consortium counts add up to more than the total.",
#     "q1_unique_consortium_count": len(unique_consortiums),
#     "q1_unique_consortiums": unique_consortiums,
#     "q2_q3_consortiums": [{
#         "consortium": r.consortium, "collection_count": int(r.collection_count),
#         "data_center_count": int(r.data_center_count),
#         "data_centers": [{
#             "dataCenter": d,
#             "collection_count": int(n)
#         } for d, n in consortium_datacenter.loc[consortium_datacenter["consortium"] == r.consortium, ["dataCenter", "collection_count"]].itertuples(index=False)]
#     } for r in consortium_summary.itertuples()],
#     "q3_data_centers": [{
#         "dataCenter": r.dataCenter, "collection_count": int(r.collection_count),
#         "consortiums": r.consortiums
#     } for r in datacenter_summary.itertuples()],
#     "q4_eosdis": {
#         "collection_count": int(eosdis["collection_count"].sum()),
#         "data_center_count": len(eosdis),
#         "data_centers": eosdis.drop(columns="graphql_server_count").to_dict(orient="records")
#     },
#     "q5_eosdis_collections": {"count": len(eosdis_coll), "file": "eosdis_collections.json"},
#     "q6_metadata_associations": {
#         "with_count": len(with_assoc), "without_count": len(without_assoc),
#         "with_file": "eosdis_collections_w_metadata_associations.json",
#         "without_file": "eosdis_collections_wo_metadata_associations.json",
#         "by_data_center": assoc_by_dc.to_dict(orient="records"),
#         "association_types": assoc_types.to_dict(orient="records")
#     },
#     "validation": {
#         "consortium_counts": check.to_dict(orient="records"),
#         "eosdis_data_center_counts": eosdis[["dataCenter", "collection_count", "graphql_server_count"]].to_dict(orient="records"),
#         "metadata_associations_split_vs_search_api": split_check
#     },
# }
# write_json(p("consortiums_datacenters_associations.json"), payload)

for f in sorted(OUT_DIR.glob("*.[cj]s*")):
    print(f"{f.name:<58} {f.stat().st_size / 1e6:7.2f} MB")
log("DONE")

eosdis_collections_w_metadata_associations.csv                6.46 MB
eosdis_collections_w_metadata_associations.json               7.91 MB
eosdis_collections_wo_metadata_associations.csv              57.78 MB
eosdis_collections_wo_metadata_associations.json             70.77 MB
eosdis_datacenter_associations_summary.csv                    0.00 MB
DONE
